# Week 2 — Sensor Distributions Across Failure Subtypes
## Member 2 — ML Engineer

## 1. Environment Setup & Imports
## 2. Load Dataset
## 3. Box Plots — Sensors Grouped by Failure Subtype
## 4. Findings — Strongest Failure Signal

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv('../data/ai4i2020.csv')

print("Dataset Shape:", df.shape)
print(df.head())

## 3. Box Plots — Sensors Grouped by Failure Subtype

In [ ]:
sensor_cols = ['Air temperature [K]', 'Process temperature [K]', 
               'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
failure_types = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

fig, axes = plt.subplots(5, 5, figsize=(20, 18))

for i, ftype in enumerate(failure_types):
    for j, sensor in enumerate(sensor_cols):
        sns.boxplot(x=ftype, y=sensor, data=df, ax=axes[i, j])
        axes[i, j].set_title(f'{sensor}\nby {ftype}', fontsize=9)
        axes[i, j].set_xlabel(ftype)
        axes[i, j].set_ylabel('')

plt.suptitle('Sensor Distributions by Failure Subtype', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Print failure type counts
print("Failure Type Counts:")
for ftype in failure_types:
    print(f"{ftype}: {df[ftype].sum()} cases")

## 4. Findings — Strongest Failure Signal

In [ ]:
# Calculate signal strength for each failure type
# Using normalized mean difference between failure=1 and failure=0 groups
signal_strength = {}

for ftype in failure_types:
    total_diff = 0
    for sensor in sensor_cols:
        group0 = df[df[ftype] == 0][sensor]
        group1 = df[df[ftype] == 1][sensor]
        if len(group1) > 0:
            mean_diff = abs(group1.mean() - group0.mean())
            std_pooled = df[sensor].std()
            normalized_diff = mean_diff / std_pooled
            total_diff += normalized_diff
    signal_strength[ftype] = total_diff

print("Signal Strength by Failure Type (sum of normalized mean differences):")
for ftype, strength in sorted(signal_strength.items(), key=lambda x: x[1], reverse=True):
    print(f"{ftype}: {strength:.4f}")

strongest = max(signal_strength, key=signal_strength.get)
print(f"\nStrongest sensor signal: {strongest}")

## Commentary — Failure Subtype Analysis

**Signal Strength Ranking (strongest to weakest):**
1. **OSF (Overstrain Failure)** — 4.57 (strongest signal)
2. **HDF (Heat Dissipation Failure)** — 4.31
3. **TWF (Tool Wear Failure)** — 2.34
4. **PWF (Power Failure)** — 2.30
5. **RNF (Random Failure)** — 1.85 (weakest signal)

**Key Observations:**
- **OSF** shows the clearest separation in sensor values — likely driven by combination of high **Torque** and high **Tool wear**, since overstrain occurs when torque × tool wear exceeds a critical threshold
- **HDF** is closely related to **temperature differences** (Air temperature vs Process temperature) and **Rotational speed**
- **RNF (Random Failures)** show the weakest sensor signal — as expected, since these are designed to be random/unpredictable failures not